# nb168 — Boltz2 on 200 calibration train compounds (P100 GPU)

Same winning recipe as nb166. 200 compounds stratified by pEC50 quintile to build a calibration mapping pEC50 ↔ boltz_affinity_pred_value.

Expected runtime: 200 × ~1min = ~3.5h on P100.

In [ ]:
import os, sys, time, subprocess, json, urllib.request, csv
from pathlib import Path

def W(msg):
    with open('/kaggle/working/trace.log', 'a') as f:
        f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)

W('=== nb168 START ===')

import torch
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
W(f'system torch={torch.__version__} cc={cc}')
need_downgrade = cc[0] < 7

# Load calibration set via direct CSV in repo (uploaded to dataset)
# Fallback: download from HF + filter to stratified subset
HF_TRAIN = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main/pxr-challenge_TRAIN.csv'
train_csv = '/kaggle/working/train.csv'
if not Path(train_csv).exists():
    urllib.request.urlretrieve(HF_TRAIN, train_csv)

compounds = []
with open(train_csv) as f:
    rows = list(csv.DictReader(f))
# Sort by pEC50, take stratified 40 per quintile
rows = sorted(rows, key=lambda r: float(r['pEC50']))
n = len(rows)
indices = []
for q in range(5):
    lo, hi = q * n // 5, (q + 1) * n // 5
    quintile = rows[lo:hi]
    step = max(1, len(quintile) // 40)
    indices.extend(quintile[::step][:40])
for r in indices:
    compounds.append({'name': r['Molecule Name'], 'smiles': r['SMILES'], 'pec50': float(r['pEC50'])})
W(f'Selected {len(compounds)} calibration compounds (stratified)')
W(f'  pEC50 range: {min(c["pec50"] for c in compounds):.2f} to {max(c["pec50"] for c in compounds):.2f}')

In [ ]:
# Write YAMLs BEFORE install (avoid pandas/numpy break)
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
YDIR = Path('/kaggle/working/yamls'); YDIR.mkdir(exist_ok=True)
ODIR = Path('/kaggle/working/outs'); ODIR.mkdir(exist_ok=True)
def safe(n): return ''.join(c if c.isalnum() else '_' for c in str(n))
for c in compounds:
    s = safe(c['name'])
    yfile = YDIR / f'{s}.yaml'
    if not yfile.exists():
        yfile.write_text(
            f'version: 1\n'
            f'sequences:\n'
            f'- protein:\n    id: A\n    sequence: {PXR_SEQ}\n'
            f'- ligand:\n    id: B\n    smiles: {c["smiles"]}\n'
            f'properties:\n- affinity:\n    binder: B\n'
        )
W(f'Wrote {len(compounds)} YAMLs')

In [ ]:
W('=== INSTALL ===')
if need_downgrade:
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                        'torch==2.4.0', 'torchvision==0.19.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu121'],
                       capture_output=True, text=True, timeout=1200)
    W(f'  torch downgrade rc={r.returncode} elapsed={time.time()-t0:.0f}s')
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boltz'],
                   capture_output=True, text=True, timeout=1200)
W(f'  boltz install rc={r.returncode} elapsed={time.time()-t0:.0f}s')
W('=== INSTALL OK ===')

In [ ]:
env_clean = {**os.environ, 'PYTHONNOUSERSITE': '1'}
def find_aff(d):
    for jf in Path(d).rglob('*affinity*.json'):
        try:
            j = json.load(open(jf))
            return {'affinity_pred_value': j.get('affinity_pred_value'),
                    'affinity_probability_binary': j.get('affinity_probability_binary')}
        except Exception: pass
    return {}

results = []
results_csv = '/kaggle/working/nb168_train_calibration.csv'
def flush():
    with open(results_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['name','smiles','pec50','affinity_pred_value','affinity_probability_binary'])
        w.writeheader()
        for r in results: w.writerow(r)

t_start = time.time()
for i, c in enumerate(compounds):
    name = c['name']
    s = safe(name)
    out_p = ODIR / s
    aff = find_aff(out_p) if out_p.exists() else {}
    if not aff.get('affinity_pred_value'):
        cmd = ['boltz', 'predict', str(YDIR / f'{s}.yaml'), '--out_dir', str(out_p),
               '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
        try:
            subprocess.run(cmd, env=env_clean, capture_output=True, text=True, timeout=900)
            aff = find_aff(out_p)
        except Exception:
            aff = {}
    results.append({
        'name': name, 'smiles': c['smiles'], 'pec50': c['pec50'],
        'affinity_pred_value': aff.get('affinity_pred_value'),
        'affinity_probability_binary': aff.get('affinity_probability_binary')
    })
    if (i + 1) % 5 == 0:
        elapsed = (time.time() - t_start) / 60
        n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
        W(f'  {i+1}/{len(compounds)} elapsed={elapsed:.0f}min non_nan={n_ok}/{len(results)}')
        flush()
flush()
n_ok = sum(1 for r in results if r.get('affinity_pred_value') is not None)
W(f'=== DONE: {n_ok}/{len(results)} calibration compounds with boltz affinity ===')
if n_ok > 5:
    import statistics
    valid = [(r['pec50'], r['affinity_pred_value']) for r in results if r['affinity_pred_value'] is not None]
    pecs = [v[0] for v in valid]
    affs = [v[1] for v in valid]
    n = len(valid)
    sx = sum(affs); sy = sum(pecs); sxy = sum(a*p for a,p in zip(affs,pecs)); sxx = sum(a*a for a in affs)
    slope = (n*sxy - sx*sy) / (n*sxx - sx*sx) if (n*sxx - sx*sx) != 0 else 0
    intercept = (sy - slope*sx) / n
    W(f'  Linear: pEC50 = {slope:.4f} * boltz_aff + {intercept:.4f}')
    correlation = (n*sxy - sx*sy) / ((n*sxx - sx*sx)**0.5 * (n*sum(p*p for p in pecs) - sy*sy)**0.5)
    W(f'  Pearson r = {correlation:.4f}')